# 01 — 因子研究

**前置：** 已完成 `00_data_download_and_sync.ipynb`（存在 `data/session_config.json`）。

**职责：**
- 单因子 IC / ICIR 有效性排名
- 因子相关性热图（识别冗余）
- Label 分布对比（absret / excess / rank）
- 输出 `data/factor_selection.json` 供训练时参考

**注意：** 本 notebook 只用测试集做因子验证，结论不泄漏到训练。

## ⚙️ 读取 session 配置（也可手动覆盖）

In [ ]:
import sys, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
warnings.filterwarnings('ignore')

ROOT = Path.cwd()
while not (ROOT / 'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

# 读取 00 生成的 session 配置
cfg_path = ROOT / 'data' / 'session_config.json'
if cfg_path.exists():
    cfg = json.loads(cfg_path.read_text())
    MARKET, SYMBOLS, BENCHMARK = cfg['market'], cfg['symbols'], cfg['benchmark']
    TRAIN_START, TRAIN_END = cfg['train_start'], cfg['train_end']
    TEST_START, TEST_END = cfg['test_start'], cfg['test_end']
    print(f'✅ Loaded session_config.json ({cfg["created_at"][:19]})')
else:
    # fallback: 手动填写
    print('⚠️ session_config.json not found, using defaults')
    MARKET, SYMBOLS, BENCHMARK = 'us', ['AAPL','NVDA','MSFT','GOOGL','AMZN','META','TSLA','AVGO','COST','NFLX'], 'QQQ'
    TRAIN_START, TRAIN_END = '2021-01-01', '2024-12-31'
    TEST_START, TEST_END   = '2025-01-01', '2026-06-18'

print(f'Market={MARKET.upper()}  Symbols={len(SYMBOLS)}  Test={TEST_START}→{TEST_END}')

from src.common.qlib_init import build_qlib_init_cfg, safe_qlib_init
from src.core.metrics import compute_ic_series

safe_qlib_init(build_qlib_init_cfg(None, market=MARKET))
from qlib.data import D
from qlib.contrib.data.loader import Alpha158DL

## 1. Label 分布对比 — absret / excess / rank

In [ ]:
import time

t0 = time.perf_counter()
y_raw = D.features(SYMBOLS, ['Ref($close,-10)/Ref($close,-1)-1'],
                   start_time=TEST_START, end_time=TEST_END)
y_base = y_raw.iloc[:,0]
print(f'Label loaded in {time.perf_counter()-t0:.1f}s  shape={y_raw.shape}')

labels = {
    'absret': y_base.copy(),
    'excess': y_base - y_base.groupby(level=0).transform('mean').fillna(0),
    'rank':   y_base.groupby(level=0).rank(pct=True),
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, s) in zip(axes, labels.items()):
    ax.hist(s.dropna(), bins=80, color='#6366f1', alpha=0.8, edgecolor='white')
    ax.axvline(0, color='red', ls='--', lw=0.8)
    ax.axvline(s.mean(), color='green', ls='--', lw=0.8, label=f'mean={s.mean():.4f}')
    ax.set_title(f'Label: {name}'); ax.legend(fontsize=8)
plt.suptitle('Label Distribution Comparison', fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

print('Label stats:')
for name, s in labels.items():
    print(f'  {name:8s}: mean={s.mean():.4f}  std={s.std():.4f}  skew={s.skew():.2f}')

## 2. 因子加载 — Alpha158 + 自定义

In [ ]:
# 定义要研究的因子组（可随时增减）
FACTOR_GROUPS = {
    'price_momentum': [
        '$close/Ref($close,5)-1',
        '$close/Ref($close,10)-1',
        '$close/Ref($close,20)-1',
        '$close/Ref($close,60)-1',
    ],
    'volatility': [
        'Std($close,10)',
        'Std($close,20)',
        'Std($ret,10)',
    ],
    'volume': [
        '$volume/Ref($volume,10)-1',
        '$volume/Mean($volume,20)-1',
    ],
    'alpha158_subset': [
        # 从 Alpha158 中选出常见的几个做对比
        'Ref($close,1)/Ref($open,1)-1',   # overnight gap
        '$high/$low-1',                    # daily range
        'Mean($close,5)/Mean($close,20)-1', # 5/20 MA ratio
    ],
}

all_factors = [f for grp in FACTOR_GROUPS.values() for f in grp]
print(f'Total factors to research: {len(all_factors)}')

t0 = time.perf_counter()
X_factors = D.features(SYMBOLS, all_factors, start_time=TEST_START, end_time=TEST_END)
X_factors = X_factors.fillna(0.0).replace([np.inf, -np.inf], 0.0)
# 简化列名用于显示
short_names = {f: f.replace('$','').replace('(','_').replace(')','').replace(',','_').replace('/','_d_')[:25]
               for f in all_factors}
X_factors.columns = [short_names[f] for f in all_factors]
print(f'Loaded in {time.perf_counter()-t0:.1f}s  shape={X_factors.shape}')

## 3. 单因子 IC / ICIR 排名

In [ ]:
LABEL_FOR_IC = 'absret'   # 用哪个 label 计算 IC
y_ic = labels[LABEL_FOR_IC].to_frame('return')

ic_results = []
for col in X_factors.columns:
    factor_panel = X_factors[[col]].rename(columns={col: 'score'})
    ic = compute_ic_series(factor_panel, y_ic, min_stocks=3)
    ic_results.append({
        'factor': col,
        'ic_mean': ic['ic_mean'], 'ic_ir': ic['ic_ir'],
        'ic_pos_pct': ic['ic_pos_pct'], 'n_days': ic['n_days'],
    })
    print(f'  {col:30s}: IC={ic["ic_mean"]:.4f}  IR={ic["ic_ir"]:.3f}  pos={ic["ic_pos_pct"]:.1%}')

ic_df = pd.DataFrame(ic_results).sort_values('ic_ir', ascending=False)
print(f'\n=== Top 5 by ICIR ===')
print(ic_df.head().to_string(index=False))
print(f'\n=== Bottom 5 by ICIR ===')
print(ic_df.tail().to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# IC Mean 横向柱状图
colors = ['#22c55e' if v > 0 else '#ef4444' for v in ic_df['ic_mean']]
axes[0].barh(range(len(ic_df)), ic_df['ic_mean'], color=colors)
axes[0].set_yticks(range(len(ic_df)))
axes[0].set_yticklabels(ic_df['factor'], fontsize=7)
axes[0].axvline(0, color='black', lw=0.5)
axes[0].set_title(f'Single-Factor IC Mean (label={LABEL_FOR_IC})')
axes[0].set_xlabel('IC Mean')

# ICIR 横向柱状图
colors2 = ['#22c55e' if v > 0 else '#ef4444' for v in ic_df['ic_ir']]
axes[1].barh(range(len(ic_df)), ic_df['ic_ir'], color=colors2)
axes[1].set_yticks(range(len(ic_df)))
axes[1].set_yticklabels(ic_df['factor'], fontsize=7)
axes[1].axvline(0, color='black', lw=0.5)
axes[1].axvline(0.3, color='green', ls='--', lw=0.8, label='IR=0.3 threshold')
axes[1].axvline(-0.3, color='red', ls='--', lw=0.8)
axes[1].legend(fontsize=8)
axes[1].set_title('Single-Factor ICIR')

plt.tight_layout(); plt.show()

## 4. 因子相关性热图 — 剔除冗余

In [ ]:
corr = X_factors.corr(method='spearman')

fig, ax = plt.subplots(figsize=(max(8, len(corr)*0.6), max(6, len(corr)*0.5)))
im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(len(corr.columns))); ax.set_xticklabels(corr.columns, rotation=45, ha='right', fontsize=7)
ax.set_yticks(range(len(corr.index)));   ax.set_yticklabels(corr.index, fontsize=7)
for i in range(len(corr)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f'{corr.iloc[i,j]:.2f}', ha='center', va='center', fontsize=6,
                color='white' if abs(corr.iloc[i,j]) > 0.7 else 'black')
plt.colorbar(im, ax=ax)
ax.set_title('Factor Correlation Matrix (Spearman)', fontweight='bold')
plt.tight_layout(); plt.show()

# 高相关对提示
high_corr = []
for i in range(len(corr)):
    for j in range(i+1, len(corr.columns)):
        if abs(corr.iloc[i,j]) > 0.8:
            high_corr.append((corr.index[i], corr.columns[j], corr.iloc[i,j]))
if high_corr:
    print(f'⚠️ High correlation pairs (|r|>0.8) — consider removing one:')
    for f1, f2, r in sorted(high_corr, key=lambda x: -abs(x[2])):
        print(f'  {f1} × {f2}: r={r:.3f}')
else:
    print('✅ No high-correlation pairs')

## 5. 保存因子选择结论

In [ ]:
from datetime import datetime

# 自动建议：IC>0 且 ICIR>0.1 的因子
good_factors = ic_df[(ic_df['ic_mean'] > 0) & (ic_df['ic_ir'] > 0.1)]['factor'].tolist()
weak_factors = ic_df[(ic_df['ic_mean'].abs() < 0.01) | (ic_df['ic_ir'].abs() < 0.05)]['factor'].tolist()

factor_sel = {
    'created_at': datetime.now().isoformat(),
    'market': MARKET, 'label': LABEL_FOR_IC,
    'good_factors': good_factors,     # 建议保留
    'weak_factors': weak_factors,     # 建议剔除
    'ic_summary': ic_df.to_dict(orient='records'),
}
sel_path = ROOT / 'data' / 'factor_selection.json'
sel_path.write_text(json.dumps(factor_sel, indent=2, default=str))

print(f'✅ Factor selection saved → {sel_path}')
print(f'  Good factors ({len(good_factors)}): {good_factors}')
print(f'  Weak factors ({len(weak_factors)}): {weak_factors}')
print('─' * 60)
print('下一步: end_to_end_training_pipeline.ipynb')
print('  它会自动读取 session_config.json 和 factor_selection.json')